In [0]:
# ML-Bibliotheken aus PySpark importieren
from pyspark.ml.feature import VectorAssembler # Kombiniert mehrere Spalten zu einem Feature-Vektor
from pyspark.ml.regression import RandomForestRegressor # Random Forest Algorithmus für Regression
from pyspark.ml.evaluation import RegressionEvaluator # Bewertet das Modell mit Metriken (RMSE, MAE)

# Quelltabelle aus dem Gold Layer definieren
source_table = 'citibike_lakehouse.gold.hourly_demand'

# Tabelle als Spark DataFrame laden
df = spark.table(source_table)

# Daten und Schema anzeigen zur ersten Überprüfung
display(df)
df.printSchema()

In [0]:
# Nur relevante Spalten auswählen
ml_df = (
    df.select(
        "hour",
        "day_of_week",
        "trip_count"
    )
    .dropna() # Zeilen mit fehlenden Werten entfernen
)

display(ml_df)

In [0]:
# VectorAssembler erstellen: kombiniert 'hour' und 'day_of_week' zu einem Vektor z.B. hour=9, day_of_week=2 → features=[9.0, 2.0]
assembler = VectorAssembler(
    inputCols=["hour", "day_of_week"], # Eingabespalten
    outputCol="features" # Name der neuen Vektorspalte
)

# Transformation anwenden → neue Spalte 'features' wird hinzugefügt
ml_read_df = assembler.transform(ml_df)

display(ml_read_df)

In [0]:
# Daten in Trainings- (80%) und Testdaten (20%) aufteilen, seed=42 sorgt für Reproduzierbarkeit – gleiche Aufteilung bei jedem Ausführen
train_df, test_df = ml_read_df.randomSplit([0.8, 0.2], seed=42)

print(f"Train rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

In [0]:
# Random Forest Regressor konfigurieren
rf = RandomForestRegressor(
    featuresCol="features", # Spalte mit dem Feature-Vektor
    labelCol="trip_count", # Zielspalte, die vorhergesagt werden soll
    predictionCol="prediction", # Name der neuen Vorhersage-Spalte
    numTrees=50, # Anzahl der Entscheidungsbäume im Wald
    maxDepth=5, # Maximale Tiefe jedes Baumes
    seed=42 # Seed für Reproduzierbarkeit
)

# Modell auf Trainingsdaten trainieren
model = rf.fit(train_df)

In [0]:
# Modell auf Testdaten anwenden → fügt 'prediction'-Spalte hinzu
predictions = model.transform(test_df)

# Echte Werte und Vorhersagen nebeneinander anzeigen
display(
    predictions.select(
        "hour",
        "day_of_week",
        "trip_count",
        "prediction"
    )
)

In [0]:
# RMSE-Evaluator erstellen (bestraft große Fehler stärker)
rmse_evaluator = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="rmse" # Root Mean Squared Error
)

# MAE-Evaluator erstellen (durchschnittlicher absoluter Fehler)
mae_evaluator = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="mae" # Mean Absolute Error
)

# Metriken berechnen und ausgeben
rmse = rmse_evaluator.evaluate(predictions)
mae = mae_evaluator.evaluate(predictions)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS citibike_lakehouse.ml;

CREATE VOLUME IF NOT EXISTS citibike_lakehouse.ml.model_artifacts;

In [0]:
model_path = "/Volumes/citibike_lakehouse/ml/model_artifacts/bike_demand_random_forest"

# Modell speichern (overwrite überschreibt ggf. vorhandenes Modell)
model.write().overwrite().save(model_path)

print(f"Model saved to {model_path}")